# Customer Support Agent — Evaluation, LangSmith Observability & OpenTelemetry

**Goal:** Build a small e-commerce customer support agent with **LangGraph**, trace every run in **LangSmith**, and learn how to evaluate it like a real ML system.

By the end of this notebook you will have:
1. A LangGraph agent that classifies support tickets into clear categories
2. A human-labeled ticket dataset loaded from CSV (`Ticket text` + `True category`)
3. LangSmith traces for every prediction (clickable in the UI), emitted through OpenTelemetry
4. Accuracy / precision / recall metrics
5. A CSV of results you can open in a spreadsheet and annotate
6. A simple loop: **inspect failures → tweak the prompt → re-run → compare**

---

## The workflow you'll follow

1. **Run the agent** on the labeled tickets.
2. **Look at the metrics** and open the exported CSV in Google Sheets / Excel.
3. **Add validator comments** in the `validator_comment` column — flag anything that looks wrong, ambiguous, or surprising.
4. **Cluster the failures** into 2–4 *failure categories* (e.g., "misroutes complaints as questions", "confuses refund vs. return").
5. **Pick the one failure category that matters most** for your use case.
6. **Tweak the classifier prompt** at the bottom of the notebook to address it.
7. **Re-run** the agent and compare the new metrics + LangSmith/OpenTelemetry traces.

## 1. Install dependencies

In [3]:
# --- Dependencies -----------------------------------------------------------
# Run once. Prints which interpreter the kernel is using, then installs only
# the packages that are missing -- with pip output VISIBLE so the cell can
# never silently "hang".
#
# Why not `%pip install --quiet ...`?
#  * `--quiet` hides all download progress. If the kernel points at an empty
#    environment, a multi-hundred-MB install (scipy, scikit-learn, grpcio,
#    ...) then looks frozen for minutes and can even OOM-restart the kernel.
#  * `--only-binary=:all:` below stops pip ever compiling from source, the
#    other common cause of an install that hangs forever. If it errors on a
#    package, re-run this cell without that flag.
#  * Still slow? Run the same `pip install` in a TERMINAL, not the notebook --
#    terminals show progress bars and let you Ctrl-C.

import importlib.util
import subprocess
import sys

print("Kernel interpreter:", sys.executable)
print("Python version:    ", sys.version.split()[0])

PACKAGES = {
    # import name -> pip requirement
    "langgraph": "langgraph",
    "langsmith": "langsmith[otel]>=0.4.25",
    "langchain_openai": "langchain-openai",
    "langchain_core": "langchain-core",
    "pandas": "pandas",
    "sklearn": "scikit-learn",
    "pydantic": "pydantic",
    "tqdm": "tqdm",
    "opentelemetry.sdk": "opentelemetry-sdk",
    "opentelemetry.exporter.otlp": "opentelemetry-exporter-otlp",
}

missing = [req for mod, req in PACKAGES.items() if importlib.util.find_spec(mod) is None]

if not missing:
    print()
    print("All dependencies already present - nothing to install.")
else:
    print()
    print("Installing:", ", ".join(missing))
    subprocess.run(
        [sys.executable, "-m", "pip", "install",
         "--only-binary=:all:", "--disable-pip-version-check", *missing],
        check=True,
    )
    print()
    print("Done. If anything was installed, restart the kernel before continuing.")

Kernel interpreter: c:\Users\SP\AppData\Local\Programs\Python\Python313\python.exe
Python version:     3.13.12

All dependencies already present - nothing to install.


## 2. Set up API keys, LangSmith tracing, and OpenTelemetry

You need two keys:
- **`OPENAI_API_KEY`** — get one at [platform.openai.com/api-keys](https://platform.openai.com/api-keys)
- **`LANGSMITH_API_KEY`** — get one at [smith.langchain.com](https://smith.langchain.com) → Settings → API Keys

Once these are set, every LLM call is traced in LangSmith. We also enable OpenTelemetry so each ticket-level eval run can carry standard span metadata like ticket id, prompt version, expected label, predicted label, and correctness.

In [4]:
import os
import getpass

# Clear stale tracing variables so an old endpoint does not make OTel export to a 404 URL.
for k in [
    "LANGCHAIN_API_KEY",
    "LANGCHAIN_ENDPOINT",
    "LANGCHAIN_TRACING_V2",
    "OTEL_EXPORTER_OTLP_ENDPOINT",
    "OTEL_EXPORTER_OTLP_TRACES_ENDPOINT",
    "OTEL_EXPORTER_OTLP_HEADERS",
    "OTEL_EXPORTER_OTLP_TRACES_HEADERS",
    "OTEL_EXPORTER_OTLP_PROTOCOL",
]:
    os.environ.pop(k, None)

def _set(key: str, prompt: str):
    if not os.environ.get(key):
        os.environ[key] = getpass.getpass(prompt)

_set("OPENAI_API_KEY",    "OpenAI API key: ")
_set("LANGSMITH_API_KEY", "LangSmith API key: ")

# LangSmith is still the evaluation/debugging UI.
os.environ["LANGSMITH_TRACING"]  = "true"
os.environ["LANGSMITH_PROJECT"]  = "customer-support-evals"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"  # use https://eu.api.smith.langchain.com for EU accounts

# OpenTelemetry is the standard tracing layer. LangSmith receives the emitted spans.
os.environ["LANGSMITH_OTEL_ENABLED"] = "true"
os.environ["OTEL_SERVICE_NAME"] = "customer-support-evals-notebook"
os.environ["OTEL_EXPORTER_OTLP_ENDPOINT"] = "https://api.smith.langchain.com/otel"
os.environ["OTEL_EXPORTER_OTLP_TRACES_ENDPOINT"] = "https://api.smith.langchain.com/otel/v1/traces"
os.environ["OTEL_EXPORTER_OTLP_PROTOCOL"] = "http/protobuf"

otel_headers = f"x-api-key={os.environ['LANGSMITH_API_KEY']},Langsmith-Project={os.environ['LANGSMITH_PROJECT']}"
os.environ["OTEL_EXPORTER_OTLP_HEADERS"] = otel_headers
os.environ["OTEL_EXPORTER_OTLP_TRACES_HEADERS"] = otel_headers

print("Traces will appear in LangSmith project:", os.environ["LANGSMITH_PROJECT"])
print("OpenTelemetry service name:", os.environ["OTEL_SERVICE_NAME"])
print("OpenTelemetry traces endpoint:", os.environ["OTEL_EXPORTER_OTLP_TRACES_ENDPOINT"])


Traces will appear in LangSmith project: customer-support-evals
OpenTelemetry service name: customer-support-evals-notebook
OpenTelemetry traces endpoint: https://api.smith.langchain.com/otel/v1/traces


## 3. Define the ticket categories

We keep the label space small and unambiguous on purpose — when there are 50 categories, *everything* looks like a model failure. Five is a good teaching size.

| Category | What belongs here |
|---|---|
| `order_status` | "Where is my order?", tracking, delivery ETA |
| `refund_request` | Customer wants money back, return-for-refund |
| `product_issue` | Item arrived broken, wrong, defective, or not as described |
| `account_help` | Login, password, address, payment method changes |
| `other` | Anything that doesn't fit above (general questions, feedback) |

In [6]:
CATEGORIES = [
    "order_status",
    "refund_request",
    "product_issue",
    "account_help",
    "other",
]

## 4. Load the labeled ticket dataset

We load a human-labeled dataset from `Week 4_ AI Evals Data.csv`. Each row is a support message with a **ground-truth category** assigned by a human annotator.

| CSV column | Used as |
|---|---|
| `Ticket text` | the message the agent classifies (`text`) |
| `True category` | the ground-truth label we score against (`true_category`) |

The other columns (`Predicted category`, `Reasoning`, `Pass / Fail`, `Annotation`) are from a previous eval pass and are ignored here — we regenerate predictions from scratch below. The worked `EX` example row at the top of the file is skipped.

To use a different dataset, point `DATA_PATH` at another CSV that has these two columns.

In [ ]:
import pandas as pd
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from typing import List
from collections import Counter

# --- Load the labeled ticket dataset from disk ---------------------------
# Each row is a (message, true_category) pair a human has already labeled.
#   "Ticket text"  -> text          (what the agent classifies)
#   "True category" -> true_category (the ground-truth label we score against)
DATA_PATH = "Week 4_ AI Evals Data.csv"

raw = pd.read_csv(DATA_PATH, usecols=["#", "Ticket text", "True category"])
raw = raw.rename(columns={"#": "id", "Ticket text": "text", "True category": "true_category"})

# Drop the blank trailing rows and the worked "EX" example at the top.
raw = raw.dropna(subset=["text", "true_category"])
raw = raw[raw["id"].astype(str).str.strip().str.lower() != "ex"]
raw["id"] = raw["id"].astype(str).str.strip()
raw["text"] = raw["text"].str.strip()
raw["true_category"] = raw["true_category"].str.strip()

# Sanity-check: every label must be one of the categories from step 3.
unknown = sorted(set(raw["true_category"]) - set(CATEGORIES))
assert not unknown, f"CSV has labels not in CATEGORIES: {unknown}"

tickets = [
    {"id": r["id"], "text": r["text"], "true_category": r["true_category"], "source": "labeled_csv"}
    for r in raw.to_dict("records")
]

print(f"Loaded {len(tickets)} labeled tickets from {DATA_PATH}")
for cat, n in sorted(Counter(t["true_category"] for t in tickets).items()):
    print(f"  {cat:>15}: {n}")
print()
for t in tickets[:5]:
    print(f"  [{t['true_category']:>15}] {t['text'][:90]}")

## 5a. What OpenTelemetry adds here

LangSmith is the place where we inspect AI traces and compare baseline vs improved runs. OpenTelemetry is the standard way we emit structured traces from code.

In this notebook, we add one OpenTelemetry parent span around each ticket classification. The LangGraph / LangChain internals still create child spans for the model call, and the parent span carries eval metadata:

- `eval.example_id`: ticket id
- `eval.run_name`: baseline or improved
- `eval.prompt_version`: v1 or v2
- `eval.true_category`: ground-truth label
- `eval.predicted_category`: model output
- `eval.correct`: whether the prediction matched the label
- `eval.reasoning`: model explanation

This makes each row in the CSV traceable back to a specific LangSmith/OpenTelemetry run.

If you see `Failed to export span batch code: 404`, the classifier is still running, but the OTLP exporter is pointed at the wrong URL. This notebook clears stale `OTEL_EXPORTER_*` values and explicitly sends trace spans to `https://api.smith.langchain.com/otel/v1/traces`.


## 5. Build the LangGraph classification agent

LangGraph models an agent as a **graph of nodes**. For a classifier, the graph is tiny — one node that calls the LLM with a structured output schema. We're using LangGraph here (instead of just calling the LLM directly) so the pattern scales to multi-step agents later (e.g., add a retrieval node, a tool-calling node, a confidence-check node).

Because LangSmith tracing is on, **every graph invocation becomes a clickable trace** showing each node's input/output.

In [8]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END
from langchain_core.prompts import ChatPromptTemplate

# --- THIS PROMPT IS WHAT YOU'LL TWEAK LATER ---
CLASSIFIER_PROMPT = """You are a triage system for an e-commerce support inbox.

Classify the customer's ticket into EXACTLY ONE of these categories:

- order_status: questions about where an order is, tracking, delivery ETA
- refund_request: the customer wants their money back
- product_issue: the item arrived broken, wrong, defective, or not as described
- account_help: login, password, address, payment method changes
- other: anything that doesn't fit the above (general questions, feedback, browsing)

Return only the category key.

Ticket:
{ticket_text}
"""

class Classification(BaseModel):
    category: Literal["order_status", "refund_request", "product_issue", "account_help", "other"]
    reasoning: str = Field(description="One short sentence explaining the choice.")

class AgentState(TypedDict):
    ticket_text: str
    category: str
    reasoning: str

def build_agent(prompt_template: str):
    """Compile a LangGraph agent. Re-call this any time you change the prompt."""
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0).with_structured_output(Classification)
    prompt = ChatPromptTemplate.from_template(prompt_template)

    def classify_node(state: AgentState) -> AgentState:
        result = (prompt | llm).invoke({"ticket_text": state["ticket_text"]})
        return {"ticket_text": state["ticket_text"], "category": result.category, "reasoning": result.reasoning}

    graph = StateGraph(AgentState)
    graph.add_node("classify", classify_node)
    graph.add_edge(START, "classify")
    graph.add_edge("classify", END)
    return graph.compile()

agent = build_agent(CLASSIFIER_PROMPT)

# Smoke test on one ticket.
sample = agent.invoke({"ticket_text": tickets[0]["text"], "category": "", "reasoning": ""})
print("Ticket:    ", tickets[0]["text"])
print("Predicted: ", sample["category"])
print("Reasoning: ", sample["reasoning"])

Ticket:     Hi, I ordered a blender 5 days ago and the tracking page hasn't updated. Can you tell me where it is?
Predicted:  order_status
Reasoning:  The customer is inquiring about the status and tracking of their order.


Failed to export span batch code: 404, reason: Not Found


In [9]:
from opentelemetry import trace

# This tracer creates ticket-level spans. LangSmith receives them because
# LANGSMITH_OTEL_ENABLED=true is set above.
tracer = trace.get_tracer("customer-support-evals")


Failed to export span batch code: 404, reason: Not Found


## 6. Run the agent on every ticket

Each invocation is automatically traced in LangSmith. After this cell finishes, go to your [LangSmith dashboard](https://smith.langchain.com) → project **`customer-support-evals`** → and you'll see every prediction with full input/output/latency.

In [10]:
import pandas as pd
from tqdm import tqdm

def run_predictions(agent, tickets, run_name="baseline", prompt_version="v1") -> pd.DataFrame:
    rows = []
    for t in tqdm(tickets, desc=f"Classifying {run_name}"):
        # One parent span per ticket makes the spreadsheet row traceable in LangSmith.
        with tracer.start_as_current_span("customer_support.classify_ticket") as span:
            span.set_attribute("langsmith.span.kind", "chain")
            span.set_attribute("eval.run_name", run_name)
            span.set_attribute("eval.prompt_version", prompt_version)
            span.set_attribute("eval.example_id", t["id"])
            span.set_attribute("eval.true_category", t["true_category"])
            span.set_attribute("input.ticket_text", t["text"])

            out = agent.invoke({"ticket_text": t["text"], "category": "", "reasoning": ""})
            correct = t["true_category"] == out["category"]

            span.set_attribute("eval.predicted_category", out["category"])
            span.set_attribute("eval.correct", correct)
            span.set_attribute("eval.reasoning", out["reasoning"])
            span.set_attribute("output.category", out["category"])

            rows.append({
                "id": t["id"],
                "ticket_text": t["text"],
                "true_category": t["true_category"],
                "predicted_category": out["category"],
                "reasoning": out["reasoning"],
                "correct": correct,
                "otel_run_name": run_name,
                "otel_prompt_version": prompt_version,
            })
    return pd.DataFrame(rows)

results_v1 = run_predictions(agent, tickets, run_name="baseline", prompt_version="v1")
results_v1.head(10)


Classifying baseline: 100%|██████████| 100/100 [01:23<00:00,  1.20it/s]


,id,ticket_text,true_category,predicted_category,reasoning,correct,otel_run_name,otel_prompt_version
0,t000,"Hi, I ordered a blender 5 days ago and the tra...",order_status,order_status,The customer is inquiring about the status and...,True,baseline,v1
1,t001,"Hello, I placed an order for a blender five da...",order_status,order_status,The customer is inquiring about the status and...,True,baseline,v1
2,t002,"Hey there, I ordered a blender about five days...",order_status,order_status,The customer is inquiring about the status and...,True,baseline,v1
3,t003,"I ordered a blender five days ago, and the tra...",order_status,order_status,The customer is inquiring about the status and...,True,baseline,v1
4,t004,I never received my order and I want my money ...,order_status,refund_request,The customer is requesting a refund due to not...,False,baseline,v1
5,t005,"I still haven't gotten my order, and I demand ...",order_status,refund_request,The customer is requesting a full refund due t...,False,baseline,v1
6,t006,"My order is nowhere to be found, and it's been...",order_status,refund_request,The customer is requesting a refund due to the...,False,baseline,v1
7,t007,This is ridiculous! I ordered the XYZ gadget w...,order_status,refund_request,The customer wants to cancel their order and g...,False,baseline,v1
8,t008,Order shows delivered last Tuesday but it's no...,order_status,order_status,The customer is inquiring about the status of ...,True,baseline,v1
9,t009,I don't see my package anywhere! It says it wa...,order_status,order_status,The customer is inquiring about the status and...,True,baseline,v1


Failed to export span batch code: 404, reason: Not Found


## 7. Evaluate: accuracy, precision, recall

- **Accuracy** = fraction of tickets classified correctly overall.
- **Precision (per class)** = of all the tickets the model *called* `refund_request`, how many actually were? High precision = few false alarms.
- **Recall (per class)** = of all the tickets that *truly were* `refund_request`, how many did the model catch? High recall = few misses.

**Why look at both:** a model that *always* predicts `other` will have 20% accuracy and 100% recall on `other` but 0% recall on everything else. Per-class precision/recall exposes that immediately.

In [11]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

def evaluate(df: pd.DataFrame, label: str):
    y_true = df["true_category"]
    y_pred = df["predicted_category"]
    print(f"=== {label} ===")
    print(f"Accuracy: {accuracy_score(y_true, y_pred):.2%}  ({df['correct'].sum()}/{len(df)} correct)\n")
    print("Per-class precision / recall / F1:")
    print(classification_report(y_true, y_pred, labels=CATEGORIES, zero_division=0))
    print("Confusion matrix (rows = true, cols = predicted):")
    cm = pd.DataFrame(
        confusion_matrix(y_true, y_pred, labels=CATEGORIES),
        index=CATEGORIES, columns=CATEGORIES,
    )
    print(cm)
    return accuracy_score(y_true, y_pred)

acc_v1 = evaluate(results_v1, "Run 1 — baseline prompt")

=== Run 1 — baseline prompt ===
Accuracy: 92.00%  (92/100 correct)

Per-class precision / recall / F1:
                precision    recall  f1-score   support

  order_status       1.00      0.80      0.89        20
refund_request       0.71      1.00      0.83        20
 product_issue       1.00      0.80      0.89        20
  account_help       1.00      1.00      1.00        20
         other       1.00      1.00      1.00        20

      accuracy                           0.92       100
     macro avg       0.94      0.92      0.92       100
  weighted avg       0.94      0.92      0.92       100

Confusion matrix (rows = true, cols = predicted):
                order_status  refund_request  product_issue  account_help  \
order_status              16               4              0             0   
refund_request             0              20              0             0   
product_issue              0               4             16             0   
account_help               0    

## 8. Export to spreadsheet for validator comments

This is the **human-in-the-loop** step. Open `results_v1.csv` in Google Sheets or Excel.

Each row has an empty `validator_comment` column. Your job:

1. **Filter `correct == FALSE`** to see only the failures.
2. For each failure, write a short note in `validator_comment` — e.g. *"complaint about delivery, model called it product_issue"*, *"ambiguous, I'd accept either"*, *"label is wrong, this really is `other`"*.
3. Also scan a sample of `correct == TRUE` rows — sometimes the model gets the right label for the *wrong reason*.
4. Once you've annotated, **cluster the comments into 2–4 failure categories** in a separate tab. For example:
   - *"Confuses `order_status` with `refund_request` when the customer mentions both delivery and money"*
   - *"Calls polite thank-you messages `account_help`"*
   - *"Routes 'wrong item' as `refund_request` instead of `product_issue`"*
5. **Pick the one failure category that matters most for your use case** (the one that's most expensive if it happens in production), and bring it back to step 9.

In [12]:
results_v1_export = results_v1.copy()
results_v1_export["validator_comment"] = ""
results_v1_export["failure_category"] = ""
results_v1_export.to_csv("results_v1.csv", index=False)
print("Wrote results_v1.csv — open it in Google Sheets or Excel.")
print("Columns:", list(results_v1_export.columns))

Wrote results_v1.csv — open it in Google Sheets or Excel.
Columns: ['id', 'ticket_text', 'true_category', 'predicted_category', 'reasoning', 'correct', 'otel_run_name', 'otel_prompt_version', 'validator_comment', 'failure_category']


## 9. Tweak the prompt to fix the failure category you picked

Edit `IMPROVED_PROMPT` below to address the failure you chose. Some common moves:

- **Add disambiguation rules.** *"If the customer mentions both delivery delay AND a refund request, classify as `order_status` — the refund is downstream of the delivery problem."*
- **Add a few-shot example** of the exact failure case with the correct label.
- **Tighten a category definition.** *"`account_help` is ONLY for login/password/profile issues, not for general site bugs."*
- **Force a step.** *"First identify the customer's primary intent in one sentence, then pick the category."*

Keep the change focused on the **one failure category** you picked. If you change everything, you won't know what helped.

In [13]:
IMPROVED_PROMPT = """You are a triage system for an e-commerce support inbox.

Classify the customer's ticket into EXACTLY ONE of these categories:

- order_status: questions about where an order is, tracking, delivery ETA, or non-delivery
- refund_request: the customer is asking for their money back (and the item itself is fine, or already returned)
- product_issue: the item arrived broken, wrong, defective, or not as described — even if the customer also asks for a refund as the remedy
- account_help: login, password, address, or payment method changes
- other: general questions, feedback, browsing, anything not covered above

Disambiguation rules:
1. If a damaged/wrong/defective item is mentioned, the category is product_issue — regardless of what remedy the customer asks for.
2. If the customer never received the order, the category is order_status — even if they mention wanting their money back.
3. If the customer is asking about a refund they already initiated ("where is my refund?"), the category is refund_request.
4. Site bugs that prevent checkout are account_help.

Think step by step:
1. What is the customer's *primary* problem? (one sentence)
2. Which category fits that problem best?

Ticket:
{ticket_text}
"""

agent_v2 = build_agent(IMPROVED_PROMPT)
results_v2 = run_predictions(agent_v2, tickets, run_name="improved", prompt_version="v2")

results_v2_export = results_v2.copy()
results_v2_export["validator_comment"] = ""
results_v2_export["failure_category"] = ""
results_v2_export.to_csv("results_v2.csv", index=False)

acc_v2 = evaluate(results_v2, "Run 2 — improved prompt")

Classifying improved:   0%|          | 0/100 [00:00<?, ?it/s]

Classifying improved: 100%|██████████| 100/100 [01:19<00:00,  1.26it/s]

=== Run 2 — improved prompt ===
Accuracy: 97.00%  (97/100 correct)

Per-class precision / recall / F1:
                precision    recall  f1-score   support

  order_status       0.95      0.95      0.95        20
refund_request       1.00      1.00      1.00        20
 product_issue       1.00      0.95      0.97        20
  account_help       0.91      1.00      0.95        20
         other       1.00      0.95      0.97        20

      accuracy                           0.97       100
     macro avg       0.97      0.97      0.97       100
  weighted avg       0.97      0.97      0.97       100

Confusion matrix (rows = true, cols = predicted):
                order_status  refund_request  product_issue  account_help  \
order_status              19               0              0             1   
refund_request             0              20              0             0   
product_issue              1               0             19             0   
account_help               0    

Failed to export span batch code: 404, reason: Not Found


## 10. Compare the two runs

Now look at the headline accuracy and at *which specific tickets flipped* between runs. Some will go from wrong → right (the win you were aiming for). Some may go from right → wrong (a regression you caused). This is normal — almost no prompt change is strictly Pareto-better, and the trade-offs are the most important thing to understand.

In [14]:
print(f"Accuracy v1: {acc_v1:.2%}")
print(f"Accuracy v2: {acc_v2:.2%}")
print(f"Δ          : {(acc_v2 - acc_v1):+.2%}\n")

comparison = results_v1.merge(
    results_v2[["id", "predicted_category", "reasoning", "correct"]],
    on="id", suffixes=("_v1", "_v2"),
)

flipped = comparison[comparison["predicted_category_v1"] != comparison["predicted_category_v2"]]
print(f"{len(flipped)} tickets changed prediction between runs.\n")

wins   = flipped[(~flipped["correct_v1"]) & (flipped["correct_v2"])]
losses = flipped[(flipped["correct_v1"]) & (~flipped["correct_v2"])]
print(f"Wins (wrong → right):     {len(wins)}")
print(f"Regressions (right → wrong): {len(losses)}")

flipped[["ticket_text", "true_category", "predicted_category_v1", "predicted_category_v2"]]

Accuracy v1: 92.00%
Accuracy v2: 97.00%
Δ          : +5.00%

11 tickets changed prediction between runs.

Wins (wrong → right):     8
Regressions (right → wrong): 3


,ticket_text,true_category,predicted_category_v1,predicted_category_v2
4,I never received my order and I want my money ...,order_status,refund_request,order_status
5,"I still haven't gotten my order, and I demand ...",order_status,refund_request,order_status
6,"My order is nowhere to be found, and it's been...",order_status,refund_request,order_status
7,This is ridiculous! I ordered the XYZ gadget w...,order_status,refund_request,order_status
16,Tracking link in the email just spins forever....,order_status,order_status,account_help
44,My laptop arrived damaged and I want a full re...,product_issue,refund_request,product_issue
45,I received my laptop in poor condition and I’m...,product_issue,refund_request,product_issue
46,"The laptop I ordered arrived broken, and I'm n...",product_issue,refund_request,product_issue
47,"I got my laptop today, but it’s damaged. I wou...",product_issue,refund_request,product_issue
58,"The product is in good shape, but the delivery...",product_issue,product_issue,order_status


## 11. What to take away

- **LangGraph** gave us a clean shape for the agent. Right now it's one node, but you can drop in retrieval, tools, or a self-check node without rewriting the eval harness.
- **LangSmith** turned every LLM call into an inspectable trace. OpenTelemetry added a standard parent span around each ticket so CSV rows, prompt versions, labels, and correctness are attached to the trace.
- **Accuracy alone is a trap.** Per-class precision/recall and the confusion matrix tell you *what kind* of mistakes the model is making.
- **Validator comments are the most valuable artifact in this whole notebook.** Numbers tell you *that* something is wrong; human notes tell you *what* and *why*.
- **Prompt iteration is a measure → diagnose → fix → re-measure loop.** Without the dataset and the metrics, prompt tweaks are just vibes.

### Suggested next steps
- Swap `Week 4_ AI Evals Data.csv` for anonymized **real** tickets from your inbox — same two columns, more rows.
- Upload the dataset to LangSmith as a versioned **Dataset** and use the LangSmith `evaluate()` runner so each prompt version gets a saved score.
- Add a second node to the graph — e.g., a confidence check that routes low-confidence predictions to a human queue.

## 12. LLM-as-judge: scoring outputs that have no ground-truth label

Every metric up to here compared the prediction against a label we wrote by hand. That only works because "classify into 5 buckets" has a single correct answer.

Most real agent output doesn't: a drafted reply, a summary, a multi-sentence explanation. There's no string to `==` against. **LLM-as-judge** hands the output plus a written rubric to a second LLM call and asks it to grade — cheap, fast, and consistent enough to run on every trace.

Here we grade the one thing in this notebook with no ground truth: the model's own `reasoning` sentence. A right label backed by generic boilerplate (*"The customer is inquiring about their order"*) is a fragile prediction — the judge surfaces those, and you can check whether low reasoning scores line up with the wrong-label rows from step 7.

**Caveats worth stating out loud:**
- The judge is itself an LLM with the same blind spots. Calibrate it against a handful of human grades before you trust the number.
- Use a written rubric with a fixed, anchored scale. "Rate 1–10" with no anchors is noise.
- Keep the judge a separate call from the thing being judged, so you can version and re-run it independently.
- Treat it as a *relative* signal (v1 vs v2), not an absolute measure of truth.

After running, open [LangSmith](https://smith.langchain.com) → project **`customer-support-evals`** and filter for the `customer_support.judge_reasoning` spans.


In [15]:
# --- 12. LLM-as-judge -----------------------------------------------------
# Every metric up to here compared the prediction against a label we wrote by
# hand. Here we grade the one artifact in this notebook that has NO ground
# truth: the model's own `reasoning` sentence. Is it a specific, correct
# justification for THIS ticket, or generic boilerplate that would "explain"
# any label?

from typing import Literal

class Judgement(BaseModel):
    justification_score: int = Field(
        description="1-5. Does the reasoning specifically and correctly justify the predicted "
                    "category for THIS ticket? 5 = precise and grounded in the ticket wording; "
                    "1 = generic filler or contradicts the ticket."
    )
    is_boilerplate: bool = Field(
        description="True if the reasoning is a template that would fit almost any ticket in this category."
    )
    verdict: Literal["good", "borderline", "bad"]
    critique: str = Field(description="One sentence: what is strong or weak about the reasoning.")

JUDGE_PROMPT = """You are grading a support-triage assistant's explanation of its own classification.

You are NOT re-classifying the ticket. Judge ONLY the quality of the `reasoning`:
- Is it specific to this ticket's actual wording, or generic filler?
- Does it actually support the predicted category?
- Would a human triage lead accept this as a real justification?

Ticket:
{ticket_text}

Predicted category: {predicted_category}
Model's reasoning: {reasoning}
"""

judge_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0).with_structured_output(Judgement)
judge_chain = ChatPromptTemplate.from_template(JUDGE_PROMPT) | judge_llm

def judge_reasoning(df: pd.DataFrame, run_name: str) -> pd.DataFrame:
    rows = []
    for r in tqdm(df.to_dict("records"), desc=f"Judging {run_name}"):
        # One span per judgement so each grade is traceable in LangSmith.
        with tracer.start_as_current_span("customer_support.judge_reasoning") as span:
            span.set_attribute("langsmith.span.kind", "chain")
            span.set_attribute("eval.run_name", run_name)
            span.set_attribute("eval.example_id", r["id"])

            j = judge_chain.invoke({
                "ticket_text": r["ticket_text"],
                "predicted_category": r["predicted_category"],
                "reasoning": r["reasoning"],
            })

            span.set_attribute("judge.justification_score", j.justification_score)
            span.set_attribute("judge.is_boilerplate", j.is_boilerplate)
            span.set_attribute("judge.verdict", j.verdict)

            rows.append({
                "id": r["id"],
                "predicted_category": r["predicted_category"],
                "correct": r["correct"],
                "justification_score": j.justification_score,
                "is_boilerplate": j.is_boilerplate,
                "verdict": j.verdict,
                "critique": j.critique,
            })
    return pd.DataFrame(rows)

judged_v1 = judge_reasoning(results_v1, run_name="baseline-judge")

print(f"Mean justification score: {judged_v1['justification_score'].mean():.2f} / 5")
print(f"Flagged as boilerplate:   {judged_v1['is_boilerplate'].sum()}/{len(judged_v1)}\n")

print("Judge verdict vs. label correctness (does weak reasoning predict wrong labels?):")
print(pd.crosstab(judged_v1["verdict"], judged_v1["correct"]))

weak = judged_v1[judged_v1["justification_score"] <= 2].merge(
    results_v1[["id", "ticket_text", "true_category"]], on="id"
)
print("\nTickets the judge scored <= 2 (weak reasoning):")
print(
    weak[["id", "true_category", "predicted_category", "justification_score", "critique"]].to_string(index=False)
    if len(weak) else "  (none)"
)


Judging baseline-judge:   0%|          | 0/100 [00:00<?, ?it/s]

Judging baseline-judge: 100%|██████████| 100/100 [01:43<00:00,  1.03s/it]

Mean justification score: 4.12 / 5
Flagged as boilerplate:   7/100

Judge verdict vs. label correctness (does weak reasoning predict wrong labels?):
correct     False  True 
verdict                 
borderline      0     13
good            8     79

Tickets the judge scored <= 2 (weak reasoning):
  (none)


Failed to export span batch code: 404, reason: Not Found
